# Environment setup and Google Drive mount
In this section, we connect Google Drive to access the MVTec dataset and set up a secure destination to save the results. We also install any missing dependencies.

In [ ]:
from google.colab import drive
import os
import shutil
import glob

# Mount Google Drive
drive.mount('/content/drive')

!git clone https://github.com/emanuelepietrocometti/sk-rd4ad.git
%cd /content/SK-RD4AD

!pip install -r requirements.txt

# Configuration variables (CURRENT RUN)
Set the dataset class you want to train and the name of the results folder for this iteration. Update these values for each new run.

In [ ]:
# === CONFIGURE YOUR RUN HERE ===
CLASS_NAME = "reda_dustOnValidation"
RUN_NAME = "01072026_205625_reda_dustOnValidation"

# Source and Destination Paths (Google Drive)
DRIVE_DATASET_PATH = f"/content/drive/MyDrive/Tesi/MVTec/{CLASS_NAME}"
DRIVE_RESULTS_PATH = f"/content/drive/MyDrive/Tesi/results/{RUN_NAME}"

# Local Paths (Fast Colab Storage)
LOCAL_DATASET_PATH = f"./mvtec/{CLASS_NAME}"
LOCAL_CHECKPOINT_PATH = "./checkpoints"
LOCAL_IMG_PATH = "./results_maps"

# Local dataset preparation
Deep Learning frameworks are significantly slower when reading images directly from Google Drive. This cell copies the dataset to Colab's local memory to maximize training speed.

In [ ]:
print(f"Starting to copy the {CLASS_NAME} dataset from Drive to local Colab storage...")

# Clear the local folder if it already exists to avoid overlapping data
if os.path.exists(LOCAL_DATASET_PATH):
    shutil.rmtree(LOCAL_DATASET_PATH)

# Copy the files
shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
print("Dataset successfully copied!")

# Create local output directories
os.makedirs(LOCAL_CHECKPOINT_PATH, exist_ok=True)
os.makedirs(LOCAL_IMG_PATH, exist_ok=True)

# Training phase
Execute the `main.py` script. This will use the default parameters (e.g., 200 epochs, res=3, learning rate 0.005) applied to the selected class.

In [ ]:
print(f"--- STARTING TRAINING FOR CLASS: {CLASS_NAME} ---")

!python main.py \
  --class_ $CLASS_NAME \
  --data_path ./mvtec/ \
  --save_path $LOCAL_CHECKPOINT_PATH/ \
  --img_path $LOCAL_IMG_PATH/

# Evaluation and metrics report
Run the `eval.py` script to generate the final report with AUPRO, AP-loc, F1-Score, etc., and save the heatmaps categorized by TP, TN, FP, FN. 
The script automatically detects the latest generated checkpoint.

In [ ]:
print(f"--- STARTING EVALUATION FOR CLASS: {CLASS_NAME} ---")

# Automatically locate the most recently generated .pth file in the local directory
checkpoints = glob.glob(f"{LOCAL_CHECKPOINT_PATH}/*.pth")

if not checkpoints:
    print("ERROR: No checkpoints found. The training phase might have failed.")
else:
    # Select the newest file if multiple exist
    latest_checkpoint = max(checkpoints, key=os.path.getctime)
    print(f"Using checkpoint: {latest_checkpoint}")
    
    # Create a specific subfolder for validation maps
    EVAL_IMG_PATH = os.path.join(LOCAL_IMG_PATH, "eval_report")
    os.makedirs(EVAL_IMG_PATH, exist_ok=True)

    !python eval.py \
      --class_ $CLASS_NAME \
      --data_path ./mvtec/ \
      --checkpoint_path $latest_checkpoint \
      --img_path $EVAL_IMG_PATH/

# Hyperparameter Fine-Tuning (Optional)
Run this block only if you want to execute the hyperparameter grid search.

In [ ]:
print(f"--- STARTING HYPERPARAMETER TUNING FOR CLASS: {CLASS_NAME} ---")

!python hyperparameters_fine_tuning.py \
  --class_ $CLASS_NAME \
  --data_path ./mvtec/ \
  --save_path ./fine_tuning_checkpoints/

# Save results to Google Drive
Final synchronization. This step copies all trained weights (`.pth`) and visualizations (`.png`) securely to the preconfigured Google Drive directory.

In [ ]:
print(f"Creating destination folder on Drive: {DRIVE_RESULTS_PATH}")
os.makedirs(DRIVE_RESULTS_PATH, exist_ok=True)

print("Transferring checkpoints...")
shutil.copytree(LOCAL_CHECKPOINT_PATH, os.path.join(DRIVE_RESULTS_PATH, "checkpoints"), dirs_exist_ok=True)

print("Transferring images and reports...")
shutil.copytree(LOCAL_IMG_PATH, os.path.join(DRIVE_RESULTS_PATH, "visualizations"), dirs_exist_ok=True)

# If fine-tuning was executed, save those results as well
if os.path.exists("./fine_tuning_checkpoints"):
    print("Transferring Fine-Tuning data...")
    shutil.copytree("./fine_tuning_checkpoints", os.path.join(DRIVE_RESULTS_PATH, "fine_tuning"), dirs_exist_ok=True)

print(f"✅ TRANSFER COMPLETE! All results are securely saved in: {DRIVE_RESULTS_PATH}")